# Fine-tuning Something Somebody Else Pretrained

**Session 26 · no homework grades this · the capstone may well use it**

The Hugging Face workflow is four objects — a tokenizer, a model, a dataset, and
a `Trainer` — and the failure modes are almost never in the modelling. They are
in what the tokenizer did to your text, in a download that was not pinned, and
in a number that came from the test set.

This notebook fine-tunes a small pretrained model on a small classification task
on **CPU**, in a couple of minutes. Every download is pinned by revision, and
the outputs are saved, so you can read it without running it.

**Before class:** run the first two cells once so the checkpoint is cached. A
first download during a lecture is not a plan.

In [1]:
import time

import numpy as np
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Pin the revision, not just the name. "distilbert-base-uncased" is a moving
# target; this commit is not.
CHECKPOINT = "distilbert-base-uncased"
REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"

print("transformers", transformers.__version__, "| torch", torch.__version__)
print("device: cpu (deliberately — this fits)")

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, revision=REVISION)
print("\nvocabulary size:", tokenizer.vocab_size)

transformers 5.15.1 | torch 2.5.1
device: cpu (deliberately — this fits)



vocabulary size: 30522


## 1. What the model actually reads

Before any training: the tokenizer is a lossy, opinionated transformation of
your text, and most surprises downstream start here.

In [2]:
sample = "Fine-tuning DistilBERT on a laptop CPU is unglamorous but instructive."
encoded = tokenizer(sample)

print("text     :", sample)
print("\ntokens   :", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("\nids      :", encoded["input_ids"][:12], "...")
print("round trip:", tokenizer.decode(encoded["input_ids"]))

text     : Fine-tuning DistilBERT on a laptop CPU is unglamorous but instructive.

tokens   : ['[CLS]', 'fine', '-', 'tuning', 'di', '##sti', '##lbert', 'on', 'a', 'laptop', 'cpu', 'is', 'un', '##gl', '##amo', '##rous', 'but', 'ins', '##truct', '##ive', '.', '[SEP]']

ids      : [101, 2986, 1011, 17372, 4487, 16643, 23373, 2006, 1037, 12191, 17368, 2003] ...
round trip: [CLS] fine - tuning distilbert on a laptop cpu is unglamorous but instructive. [SEP]


Three things to notice, all of which bite later:

- `[CLS]` and `[SEP]` were added. The sequence the model sees is longer than the
  one you wrote.
- "Fine-tuning" became several pieces. Sub-word tokenisation means an unusual
  word is not out-of-vocabulary, it is spelled out — which is why the vocabulary
  is thirty thousand entries rather than a million.
- The round trip is lowercase. This is an *uncased* checkpoint; capitalisation
  is gone before the model sees anything.

That last one matters: if your task depends on capitalisation — proper nouns,
acronyms, SHOUTING as sentiment — an uncased checkpoint has thrown away the
signal before training begins.

In [3]:
for text in ["ACME Corp filed a claim.", "acme corp filed a claim.",
             "unbelievably disappointing", "🙂 great"]:
    ids = tokenizer(text)["input_ids"]
    print(f"{text!r:<34} -> {len(ids):>2} tokens  "
          f"{tokenizer.convert_ids_to_tokens(ids)}")

'ACME Corp filed a claim.'         ->  9 tokens  ['[CLS]', 'ac', '##me', 'corp', 'filed', 'a', 'claim', '.', '[SEP]']
'acme corp filed a claim.'         ->  9 tokens  ['[CLS]', 'ac', '##me', 'corp', 'filed', 'a', 'claim', '.', '[SEP]']
'unbelievably disappointing'       ->  8 tokens  ['[CLS]', 'un', '##bel', '##ie', '##va', '##bly', 'disappointing', '[SEP]']
'🙂 great'                          ->  4 tokens  ['[CLS]', '[UNK]', 'great', '[SEP]']


## 2. A small, honest dataset

Twenty Newsgroups, two categories, shipped with scikit-learn — no Hub download,
and small enough to fine-tune on a CPU. Headers, footers and quotes are stripped
because otherwise the task is trivially solvable from the signature block, which
would be a leak rather than a result.

In [4]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

CATEGORIES = ["comp.graphics", "rec.sport.baseball"]
raw = fetch_20newsgroups(subset="all", categories=CATEGORIES,
                         remove=("headers", "footers", "quotes"),
                         random_state=0, shuffle=True)

texts = [t.strip() for t in raw.data]
labels = raw.target
keep = [i for i, t in enumerate(texts) if len(t.split()) >= 12]   # drop the stubs
texts = [texts[i] for i in keep]
labels = labels[keep]

# Three splits. The test set is opened once, at the end.
train_texts, tmp_texts, train_y, tmp_y = train_test_split(
    texts, labels, test_size=0.4, random_state=0, stratify=labels)
valid_texts, test_texts, valid_y, test_y = train_test_split(
    tmp_texts, tmp_y, test_size=0.5, random_state=0, stratify=tmp_y)

print(f"train {len(train_texts)}   validation {len(valid_texts)}   test {len(test_texts)}")
print("classes:", CATEGORIES)
print("\nan example:\n", train_texts[0][:300].replace("\n", " "), "...")

train 1089   validation 363   test 364
classes: ['comp.graphics', 'rec.sport.baseball']

an example:
 Well, it looks like, just as Doug trumped Tim, beating him to the net with his defensive analyses, so Tim has gotten in ahead of me.  The way I was doing it was a little different. Being me, of course, I used equivalent averages to work out how many runs a player was worth, and I calculated both rat ...


### Length matters more than it looks

`max_length` silently truncates. Check the distribution before choosing it, or
you will discard the half of each document where the answer lives.

The next cell tokenises without truncating, so the tokenizer will warn that some
documents exceed the model's 512-token limit. That warning **is the
measurement** — it is telling you the thing you are about to look at.

In [5]:
lengths = np.array([len(tokenizer(t)["input_ids"]) for t in train_texts])
for q in (50, 75, 90, 95, 99):
    print(f"  {q}th percentile: {int(np.percentile(lengths, q)):>5} tokens")
MAX_LENGTH = 256
print(f"\nwith max_length={MAX_LENGTH}, "
      f"{(lengths > MAX_LENGTH).mean():.1%} of documents are truncated")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1084 > 512). Running this sequence through the model will result in indexing errors


  50th percentile:   114 tokens
  75th percentile:   219 tokens
  90th percentile:   412 tokens
  95th percentile:   713 tokens
  99th percentile:  3089 tokens

with max_length=256, 20.4% of documents are truncated


## 3. Tokenise once, pad per batch

Padding every document to `max_length` wastes most of the compute on `[PAD]`
tokens. Dynamic padding pads each batch to its own longest member instead.

In [6]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

def encode(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

splits = {
    "train": Dataset.from_dict({"text": train_texts, "labels": train_y}),
    "validation": Dataset.from_dict({"text": valid_texts, "labels": valid_y}),
    "test": Dataset.from_dict({"text": test_texts, "labels": test_y}),
}
encoded = {name: ds.map(encode, batched=True, remove_columns=["text"])
           for name, ds in splits.items()}

collator = DataCollatorWithPadding(tokenizer=tokenizer)
batch = collator([encoded["train"][i] for i in range(8)])
print("one padded batch:", {k: tuple(v.shape) for k, v in batch.items()})

# The saving is a property of the whole epoch, not of one batch: batches of long
# documents save nothing, batches of short ones save almost everything.
BATCH = 16
rows = list(range(len(encoded["train"])))
fixed = dynamic = 0
for start in range(0, len(rows), BATCH):
    chunk = [encoded["train"][i] for i in rows[start:start + BATCH]]
    longest = max(len(r["input_ids"]) for r in chunk)
    fixed += len(chunk) * MAX_LENGTH
    dynamic += len(chunk) * longest

# And again with the rows sorted by length, so batches are internally similar.
by_length = sorted(rows, key=lambda i: len(encoded["train"][i]["input_ids"]))
grouped = 0
for start in range(0, len(by_length), BATCH):
    chunk = [encoded["train"][i] for i in by_length[start:start + BATCH]]
    grouped += len(chunk) * max(len(r["input_ids"]) for r in chunk)

print(f"\nover the whole training set, at batch size {BATCH}:")
print(f"  padding every batch to max_length : {fixed:,} token slots")
print(f"  dynamic padding, shuffled order   : {dynamic:,} ({dynamic / fixed:.0%})")
print(f"  dynamic padding, length-grouped   : {grouped:,} ({grouped / fixed:.0%})")

Map:   0%|          | 0/1089 [00:00<?, ? examples/s]

Map:   0%|          | 0/363 [00:00<?, ? examples/s]

Map:   0%|          | 0/364 [00:00<?, ? examples/s]

one padded batch: {'labels': (8,), 'input_ids': (8, 256), 'token_type_ids': (8, 256), 'attention_mask': (8, 256)}

over the whole training set, at batch size 16:
  padding every batch to max_length : 278,784 token slots
  dynamic padding, shuffled order   : 277,585 (100%)
  dynamic padding, length-grouped   : 147,344 (53%)


**Dynamic padding on its own saved nothing here** — and that is worth
understanding rather than explaining away. A fifth of these documents hit the
256-token cap, so in shuffled order almost every batch of sixteen contains at
least one document at the cap, and the batch is padded to it anyway.

Sorting by length first is what turns the idea into a saving, because then the
short documents share batches with other short documents. The cost is that
batches are no longer randomly composed, which interacts with shuffling — the
usual compromise is to sort within a large buffer and shuffle the batches.

The general lesson is the one this course keeps returning to: an optimisation
that "obviously" helps is a measurement, not an assumption.

## 4. Fine-tuning

`AutoModelForSequenceClassification` loads the pretrained body and attaches a
fresh, randomly initialised classification head — which is exactly what the
warning below is telling you. Seeing that warning is correct; *not* seeing it
would mean you had loaded someone else's head.

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT, revision=REVISION, num_labels=2)

total = sum(p.numel() for p in model.parameters())
head = sum(p.numel() for n, p in model.named_parameters() if "classifier" in n)
print(f"\nparameters: {total:,} total, {head:,} in the new head "
      f"({head / total:.2%})")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



parameters: 66,955,010 total, 592,130 in the new head (0.88%)


In [8]:
from transformers import Trainer, TrainingArguments

def compute_metrics(prediction):
    predicted = prediction.predictions.argmax(-1)
    correct = (predicted == prediction.label_ids)
    return {"accuracy": float(correct.mean())}

args = TrainingArguments(
    output_dir="hf_finetune_demo",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=25,
    report_to=[],
    seed=0,
    use_cpu=True,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=encoded["train"], eval_dataset=encoded["validation"],
                  data_collator=collator, processing_class=tokenizer,
                  compute_metrics=compute_metrics)

t0 = time.perf_counter()
trainer.train()
print(f"\nfine-tuning took {time.perf_counter() - t0:.0f}s on CPU")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.105558,0.041733,0.988981



fine-tuning took 76s on CPU


In [9]:
validation = trainer.evaluate(encoded["validation"])
print("validation:", {k: round(v, 4) for k, v in validation.items()
                      if k in ("eval_loss", "eval_accuracy")})

Training Loss,Validation Loss,Epoch,Accuracy
0.105558,0.041733,1,0.988981


validation: {'eval_loss': 0.0417, 'eval_accuracy': 0.989}


### What did the pretraining actually buy?

The honest comparison is against a model that had no pretraining at all — same
architecture, same data, same budget, random initial weights.

In [10]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(CHECKPOINT, revision=REVISION, num_labels=2)
scratch = AutoModelForSequenceClassification.from_config(config)

scratch_args = TrainingArguments(
    output_dir="hf_scratch_demo", num_train_epochs=1,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    learning_rate=3e-5, eval_strategy="no", save_strategy="no",
    logging_steps=50, report_to=[], seed=0, use_cpu=True)

scratch_trainer = Trainer(model=scratch, args=scratch_args,
                          train_dataset=encoded["train"],
                          data_collator=collator, processing_class=tokenizer,
                          compute_metrics=compute_metrics)
scratch_trainer.train()
scratch_eval = scratch_trainer.evaluate(encoded["validation"])

print(f"\npretrained then fine-tuned : {validation['eval_accuracy']:.4f}")
print(f"same architecture from scratch: {scratch_eval['eval_accuracy']:.4f}")

Step,Training Loss
50,0.725523


Training Loss,Validation Loss,Step,Accuracy
0.725523,0.679959,69,0.506887



pretrained then fine-tuned : 0.9890
same architecture from scratch: 0.5069


That gap, for one epoch of a few thousand documents, is the entire argument for
the pretraining era. The architecture is identical; only the starting weights
differ.

## 5. A baseline that is not a transformer

Before concluding that the transformer was necessary, spend thirty seconds on
TF-IDF and logistic regression. On a two-class topic task with clean text, it is
often within a point or two — and it trains in under a second.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

t0 = time.perf_counter()
baseline = make_pipeline(TfidfVectorizer(min_df=2, ngram_range=(1, 2)),
                         LogisticRegression(max_iter=2000)).fit(train_texts, train_y)
baseline_seconds = time.perf_counter() - t0

print(f"TF-IDF + logistic regression: validation accuracy "
      f"{baseline.score(valid_texts, valid_y):.4f} in {baseline_seconds:.1f}s")
print(f"fine-tuned DistilBERT       : validation accuracy "
      f"{validation['eval_accuracy']:.4f}")

TF-IDF + logistic regression: validation accuracy 0.9394 in 0.2s
fine-tuned DistilBERT       : validation accuracy 0.9890


Report both. "We fine-tuned a transformer" is not a result unless you know what
the cheap thing scored.

## 6. Inference, and the pipeline that hides all of this

In [12]:
from transformers import pipeline

classifier = pipeline("text-classification", model=trainer.model,
                      tokenizer=tokenizer, device=-1)

examples = [
    "The renderer computes ambient occlusion per fragment on the GPU.",
    "He went three for four with a double and two runs batted in.",
]
for text, result in zip(examples, classifier(examples, truncation=True, max_length=MAX_LENGTH)):
    label = CATEGORIES[int(result["label"].split("_")[-1])]
    print(f"{text[:52]:<54} -> {label}  ({result['score']:.3f})")

The renderer computes ambient occlusion per fragment   -> comp.graphics  (0.981)
He went three for four with a double and two runs ba   -> rec.sport.baseball  (0.978)


`pipeline` is three lines and hides the tokenizer, the padding, the batching and
the `argmax`. That is fine for a demo and dangerous for a deployment: it also
hides the truncation length, and it will happily cut your document in half
without saying so. Note that we passed `truncation` and `max_length` explicitly
above for exactly that reason.

## 7. The test set, once

In [13]:
test_result = trainer.evaluate(encoded["test"])
print(f"validation accuracy (used for every choice): {validation['eval_accuracy']:.4f}")
print(f"test accuracy (opened now, once)          : {test_result['eval_accuracy']:.4f}")
print(f"TF-IDF baseline on the same test set      : {baseline.score(test_texts, test_y):.4f}")

Training Loss,Validation Loss,Epoch,Accuracy
0.105558,0.045651,1,0.986264


validation accuracy (used for every choice): 0.9890
test accuracy (opened now, once)          : 0.9863
TF-IDF baseline on the same test set      : 0.9725


In [14]:
import shutil

for path in ("hf_finetune_demo", "hf_scratch_demo"):
    shutil.rmtree(path, ignore_errors=True)
print("training output directories removed")
print(f"\nreproducibility record:")
print(f"  checkpoint {CHECKPOINT}")
print(f"  revision   {REVISION}")
print(f"  transformers {transformers.__version__}, torch {torch.__version__}")
print(f"  max_length {MAX_LENGTH}, 1 epoch, lr 3e-5, seed 0, CPU")

training output directories removed

reproducibility record:
  checkpoint distilbert-base-uncased
  revision   12040accade4e8a0f71eabdb258fecc2e7e948be
  transformers 5.15.1, torch 2.5.1
  max_length 256, 1 epoch, lr 3e-5, seed 0, CPU


## What to take from this

- The tokenizer is part of the model. Check what it does to your text — casing,
  sub-words, special tokens — before you blame the training.
- Look at the token-length distribution before choosing `max_length`, and report
  what fraction you truncated.
- Pin the revision. A checkpoint name is a moving target and "we used
  distilbert-base-uncased" does not identify the weights you ran.
- Loading a pretrained body with a fresh head produces a warning. That warning is
  correct.
- Always run the cheap baseline. Sometimes it wins, and that is a finding.
- Everything is chosen on validation. The test set appears once, at the end.

## Where to go next

- **Reading, Session 26** — pretraining, BERT and GPT, and the full workflow.
- **Session 28** — using these models without fine-tuning: prompting, retrieval,
  and LoRA when fine-tuning is too expensive.
- **The capstone** — if you fine-tune, the reproducibility record printed above
  is the minimum your report needs.